# SI — colour-free oligomer vs fibril clusters of α-synuclein aggregates

**Makes up:** Figure S88, "Clustering of α-Synuclein data" (`fig:SI_AlphaSynClustering`, folder
`Oligomers_vs_Fibrils_Clustering`): `OligomerFibril_HDBSCAN_RedShare`. **a)** the aggregates of Figure 5 k-m
clustered on number of localisations (N_molecules) and length (precision-corrected major axis); **b)** the same
plane coloured by each aggregate's relative pixel 3 QE.

Addresses Reviewer 4, Comment 10: aggregate state must be defined without the colour that is then compared. Here
the clustering sees only N_molecules and length. Each aggregate's relative pixel 3 QE (R / (R + G + B) of its summed
photoelectrons, `_nile_red.aggregate_red_share`, as Figure 5m) is compared between the clusters only afterwards.

**Method:** per aggregate (>= 10 localisations), the major axis is 2 sigma along the long principal axis of its
localisations' covariance, with the mean localisation variance subtracted from each diagonal term (so it measures
the aggregate, not the localisation blur). log10(N_molecules) and log10(major axis), each standardised, are clustered
with HDBSCAN (`min_cluster_size` 12, `min_samples` 5); the cluster of fewest localisations is the oligomers, of most
the fibrils, and aggregates HDBSCAN leaves unassigned are in neither. Length was chosen over the other shape and size
measures tested (eccentricity, aspect ratio, areas, radius of gyration, circularity): it separates a compact fibril
cluster most clearly.

**Earlier version:** clustered on area and the median `photon_rate`, then compared fitted wavelengths. That
`photon_rate` was (A_B + A_G + A_R) per frame, which over-counts detected photoelectrons by a colour-dependent
factor, so it was not colour-free; with the detected photoelectron rate the area/rate clusters no longer differ in
colour.

**Data:** the aggregates written by `../Figure_5k-m_AlphaSyn.ipynb` (BioStudies `S-BIAD3210`, α-syn Nile Red PAINT,
fitted, drift-corrected, linked, fiducials removed and segmented there):
`Localisations_Undrifted_Linked_FiducialsRemoved_clusteredlocs.h5` in its data folder. **Run that notebook first,
with the same `FAST_MODE`.**

## FAST_MODE

Selects which Figure 5 k-m run to read: `True` its fast run (first TIFF only, fewer aggregates, illustrative),
`False` the full acquisition (the published figure).

---
## 0 — Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu, median_abs_deviation
from sklearn.cluster import HDBSCAN
from sklearn.metrics import roc_auc_score

# _nile_red.py lives in the parent folder (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _nile_red as nr

from pyS3M import IOFunctions, PlottingBase

plotter = PlottingBase.PublicationPlotter(dark_background=False)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: Figure 5 k-m's fast run (first TIFF). False: its full run (the published figure).

PIXEL_SIZE_NM = 69.0      # Ximea camera pixel
MIN_LOCS_FOR_SHAPE = 10   # as Figure 5 k-m
HDB = dict(min_cluster_size=12, min_samples=5)

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
# Figure 5 k-m's data folder (its notebook runs in notebooks/figures/, OUT_ROOT = outputs/Figure_5k-m_AlphaSyn)
FIG5_DATA = REPO_ROOT / "notebooks" / "figures" / "outputs" / "Figure_5k-m_AlphaSyn" / ("data_fast" if FAST_MODE else "data_full")
CLUSTERED_FILE = FIG5_DATA / "Localisations_Undrifted_Linked_FiducialsRemoved_clusteredlocs.h5"
OUT_ROOT  = Path("outputs") / "FigureSI_OligomervsFibril"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"FAST_MODE      : {FAST_MODE}")
print(f"Aggregates from: {CLUSTERED_FILE}")
print(f"Figures dir    : {FIG_DIR.resolve()}")

---
## 2 — Per-aggregate N_molecules, length and relative pixel 3 QE

In [ ]:
if not CLUSTERED_FILE.exists():
    raise FileNotFoundError(f"{CLUSTERED_FILE} not found: run ../Figure_5k-m_AlphaSyn.ipynb first (same FAST_MODE)")
# fits written before 2026-11-11 still have `photons` columns: convert them to photoelectrons
df = IOFunctions.IO_Functions(camera="ximea").add_photoelectron_columns(pd.read_hdf(CLUSTERED_FILE))
red_share = nr.aggregate_red_share(df)


def aggregate_table(df, pixel_size_nm=PIXEL_SIZE_NM, min_locs=MIN_LOCS_FOR_SHAPE):
    # One row per aggregate: N_molecules, precision-corrected major/minor axis (nm), relative pixel 3 QE
    rec = []
    for cid, g in df.groupby("cluster_id"):
        if len(g) < min_locs:
            continue
        cov = np.cov(g["xc"].to_numpy() * pixel_size_nm, g["yc"].to_numpy() * pixel_size_nm)
        cov[0, 0] = max(cov[0, 0] - np.mean(g["xc_err"].to_numpy() ** 2) * pixel_size_nm ** 2, 0.0)
        cov[1, 1] = max(cov[1, 1] - np.mean(g["yc_err"].to_numpy() ** 2) * pixel_size_nm ** 2, 0.0)
        lam_min, lam_max = np.clip(np.linalg.eigvalsh(cov), 0, None)
        rec.append(dict(cluster_id=cid, n_locs=len(g), major_nm=2 * np.sqrt(lam_max), minor_nm=2 * np.sqrt(lam_min),
                        red_share=red_share[cid]))
    return pd.DataFrame(rec)


agg = aggregate_table(df)
agg = agg[agg["major_nm"] > 0].reset_index(drop=True)   # log axis; an aggregate needs some resolved extent
print(f"{len(agg):,} aggregates with >= {MIN_LOCS_FOR_SHAPE} localisations; "
      f"N_molecules median {agg['n_locs'].median():.0f}, major axis median {agg['major_nm'].median():.0f} nm")

---
## 3 — HDBSCAN on N_molecules and length, then relative pixel 3 QE of the clusters

The clusters are fixed before relative pixel 3 QE is looked at. Each cluster's relative pixel 3 QE is summarised as
median ± robust SD (1.4826 × MAD) over its aggregates; AUC is the probability that a fibril is redder than an
oligomer (0.5: no difference). The last lines check that the split does not hinge on the HDBSCAN settings.

In [ ]:
def hdbscan_groups(agg, hdb=HDB):
    feat = np.log10(agg[["n_locs", "major_nm"]].to_numpy())
    labels = HDBSCAN(**hdb).fit((feat - feat.mean(axis=0)) / feat.std(axis=0)).labels_
    cores = sorted({k for k in labels if k >= 0}, key=lambda k: np.log10(agg["n_locs"][labels == k]).mean())
    group = np.full(len(agg), "unassigned", dtype=object)
    if len(cores) >= 2:
        group[labels == cores[0]] = "oligomer"
        group[labels == cores[-1]] = "fibril"
    return group, len(cores)


agg["group"], n_clusters = hdbscan_groups(agg)
print(f"HDBSCAN: {n_clusters} clusters; "
      + ", ".join(f"{g} {(agg['group'] == g).sum()}" for g in ("oligomer", "fibril", "unassigned")))
for g in ("oligomer", "fibril", "unassigned"):
    v = agg.loc[agg["group"] == g]
    if len(v):
        print(f"  {g:10s}: relative pixel 3 QE {v['red_share'].median():.3f} +/- "
              f"{median_abs_deviation(v['red_share'], scale='normal'):.3f} (median +/- robust SD), "
              f"N_molecules median {v['n_locs'].median():.0f}, major axis median {v['major_nm'].median():.0f} nm")
oli, fib = agg.loc[agg["group"] == "oligomer", "red_share"], agg.loc[agg["group"] == "fibril", "red_share"]
if len(oli) and len(fib):
    auc = roc_auc_score(np.r_[np.zeros(len(oli)), np.ones(len(fib))], np.r_[oli, fib])
    print(f"fibril vs oligomer: AUC {auc:.2f}, Mann-Whitney p = {mannwhitneyu(oli, fib).pvalue:.1e}")

# robustness: the same split over the HDBSCAN settings
for mcs in (8, 12, 20):
    for ms in (3, 5, 10):
        g, k = hdbscan_groups(agg, dict(min_cluster_size=mcs, min_samples=ms))
        o, f = agg["red_share"][g == "oligomer"], agg["red_share"][g == "fibril"]
        txt = (f"oligomers {len(o)} ({o.median():.3f}), fibrils {len(f)} ({f.median():.3f})" if len(f) else "one cluster")
        print(f"  min_cluster_size {mcs:2d}, min_samples {ms:2d}: {k} clusters; {txt}")

---
## 4 — Figure S88

**a)** N_molecules vs length, coloured by cluster. **b)** The same aggregates coloured by relative pixel 3 QE (the
Figure 5 colour scale: jet over the aggregates' mean ± 2.5 SD).

In [ ]:
GRID = dict(visible=True, color="gray", lw=0.5, alpha=0.5, ls="--")
COLOURS = {"unassigned": "0.75", "oligomer": "tab:blue", "fibril": "tab:red"}
_, _, RS_MIN, RS_MAX = nr.colour_scale(agg)

fig, axs = plotter.two_column_plot(nrows=1, ncols=2, width=6.61, height=2.92)
ax = axs[0]
for g in ("unassigned", "oligomer", "fibril"):
    m = agg["group"] == g
    ax.scatter(agg.loc[m, "n_locs"], agg.loc[m, "major_nm"], s=6, alpha=0.6 if g == "unassigned" else 0.8, lw=0,
               color=COLOURS[g], label=f"{g} (n={m.sum()})", zorder=1 if g == "unassigned" else 2, rasterized=True)
ax.legend(fontsize=6, loc="upper left", markerscale=1.5)

ax = axs[1]
order = agg["red_share"].argsort()   # reddest drawn last
sc = ax.scatter(agg["n_locs"].iloc[order], agg["major_nm"].iloc[order], c=agg["red_share"].iloc[order], s=6, lw=0,
                cmap="jet", vmin=RS_MIN, vmax=RS_MAX, rasterized=True)
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label(nr.PIXEL3_LABEL)

for ax, letter in zip(axs, "ab"):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"N$_{\rm molecules}$")
    ax.set_ylabel("major axis / nm")
    ax.grid(**GRID)
    ax.text(-0.2, 1.0, f"{letter})", transform=ax.transAxes, fontsize=8, fontweight="bold", va="bottom", ha="left")
axs[1].sharex(axs[0])
axs[1].sharey(axs[0])
fig.savefig(FIG_DIR / "OligomerFibril_HDBSCAN_RedShare.svg", dpi=600, format="svg", bbox_inches="tight")
plt.show()